# 대화가 길어지면 · 골라서 보내기와 요약

강의 오후 ② 뒷부분 (Academy 모듈 2 · L4 Trim and Filter · L5 Chatbot w/ Summarization · L6 External Memory)

| 칸 | 보여 주는 것 |
| --- | --- |
| ① 골라서 보내기 | State 는 그대로 두고 노드가 최근 것만 보내면 입력 토큰이 줄어든다. 대신 보내지 않은 앞 대화는 모른다 |
| ② 요약하는 튜터 | 대화가 6개를 넘으면 요약을 쓰고 오래된 메시지는 지운다. 요약 덕분에 앞 대화를 기억한다 |
| ③ 다시 켜면 | MemorySaver 는 메모리에만 저장한다. 프로그램을 다시 켜면 비어 있다 |

모델은 실습과 같은 OpenRouter 의 `qwen/qwen3-32b` 를 씁니다. 접속 정보는 `lab/.env` 에서 읽습니다.

In [ ]:
import "npm:@langchain/core@1.2.14";
import { RemoveMessage } from "npm:@langchain/core@1.2.14/messages";
import { Annotation, MessagesAnnotation, StateGraph, START, END, MemorySaver } from "npm:@langchain/langgraph@1.4.18";

// 접속 정보: 실습과 같은 lab/.env 를 읽는다 (OpenRouter · qwen/qwen3-32b). 키는 화면에 찍지 않는다
const ENV: Record<string, string> = {};
try {
  for (const line of Deno.readTextFileSync("../lab/.env").split("\n")) {
    const m = line.match(/^\s*([A-Z_]+)\s*=\s*(.*)\s*$/);
    if (m) ENV[m[1]] = m[2];
  }
} catch { /* lab/.env 가 없으면 아래 기본값 */ }
const BASE_URL = ENV.LLM_BASE_URL ?? "https://openrouter.ai/api/v1";
const API_KEY  = ENV.LLM_API_KEY  ?? "";
const MODEL    = ENV.LLM_MODEL    ?? "qwen/qwen3-32b";
const OPTIONS  = { temperature: 0 };   // 시연이 매번 같게
console.log({ BASE_URL, MODEL, 키: API_KEY ? "있음" : "없음 — lab/.env 에 LLM_API_KEY 를 넣는다" });

// LangSmith (강사 시연용): demo/.env 에 키가 있으면 실행 기록(trace)이 LangSmith 에 남는다. 실습은 lab/.env 를 써서 남지 않는다
import { traceable } from "npm:langsmith@0.11.1/traceable";
try {
  for (const line of Deno.readTextFileSync(".env").split("\n")) {
    const m = line.match(/^\s*(LANGSMITH_[A-Z_]+)\s*=\s*(?:"([^"]*)"|([^#\s]*))/);   // "따옴표 값" 은 그대로, 아니면 # 주석 앞까지
    const value = m?.[2] ?? m?.[3];
    if (m && value) Deno.env.set(m[1], value);
  }
} catch { /* demo/.env 가 없으면 기록하지 않는다 */ }
if (!Deno.env.get("LANGSMITH_API_KEY")) Deno.env.set("LANGSMITH_TRACING", "false");   // 키가 없으면 끈다
// 모델을 부르는 함수를 감싸면 trace 에 "모델 호출" 한 칸으로 남는다. 기록을 안 켜면 그냥 원래 함수로 돈다
const asLLM = <F extends (...a: any[]) => Promise<any>>(fn: F, name: string) => traceable(fn, {
  name, run_type: "llm",
  processInputs: (i: any) => ({ messages: i.args ? i.args[0] : i.input }),
  processOutputs: (o: any) => ({ choices: [{ message: typeof o.outputs === "string" ? { role: "assistant", content: o.outputs } : o }] }),
});
console.log({ LangSmith: Deno.env.get("LANGSMITH_TRACING") === "true" && Deno.env.get("LANGSMITH_API_KEY") ? `기록함 · project ${Deno.env.get("LANGSMITH_PROJECT") ?? "default"}` : "기록 안 함 (demo/.env)" });

// 그래프 그림: 노드·엣지 이름을 mermaid.ink 로 보내 PNG 로 받아 셀 출력에 띄운다. 받지 못하면 mermaid 글자로 찍는다
async function showGraph(graph: any, options: { xray?: boolean } = {}) {
  const drawable = await graph.getGraphAsync(options);   // xray: true 면 subgraph 안의 노드까지 그린다
  try {
    const png = await drawable.drawMermaidPng();
    await Deno.jupyter.display(Deno.jupyter.image(new Uint8Array(await png.arrayBuffer())));
  } catch {
    console.log(drawable.drawMermaid());
  }
}

## ① 골라서 보내기

대화가 길어지면 messages 도 길어집니다. 수십 번 주고받은 대화를 매번 전부 보내면 토큰이 많이 들고, 그만큼 느리고 비쌉니다. 그리고 언젠가는 컨텍스트 윈도우를 넘습니다.

먼저 모델을 부르는 `chat()` 입니다. 오전에 만든 것과 같습니다.

In [ ]:
// L4 · 모델 부르기: 오전에 만든 chat() 과 같다. 입력 토큰 수도 같이 돌려준다
async function callModel(messages: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, temperature: 0 }),
  });
  const data = await res.json();
  const { role, content } = data.choices[0].message;
  return { reply: { role, content: content?.trim() ?? "" }, tokens: data.usage.prompt_tokens };
}
const chat = asLLM(callModel, "chat");   // LangSmith 에 모델 호출로 남긴다

학생과 튜터가 이미 몇 번 주고받은 대화를 준비했습니다. 서론, 본론, 연결어를 물었고, 이제 결론을 묻습니다.

In [ ]:
// 학생과 이미 여러 번 주고받은 대화
const SYSTEM = { role: "system", content: "너는 영어 쓰기 튜터다. 한국어로 두세 문장으로 짧게 답한다." };
const history = [
  SYSTEM,
  { role: "user", content: "IELTS 쓰기에서 서론은 어떻게 써?" },
  { role: "assistant", content: "질문을 내 말로 바꿔 쓰고, 내 입장을 한 문장으로 밝힌다. 두세 문장이면 충분하다." },
  { role: "user", content: "본론 문단은 몇 개가 좋아?" },
  { role: "assistant", content: "보통 두 개다. 문단마다 주장 하나와 근거, 예시를 넣는다." },
  { role: "user", content: "연결어는 뭘 쓰면 좋아?" },
  { role: "assistant", content: "However, Moreover, As a result 처럼 문단과 문장 사이의 관계를 보여 주는 말을 쓴다." },
  { role: "user", content: "결론 문단은 어떻게 써?" },
];
console.log(`대화 ${history.length}개`);

먼저 지금까지 해 온 대로 messages 를 **전부** 보내 보겠습니다.

In [ ]:
// ① 전부 보내기
const State = Annotation.Root({
  messages: Annotation<any[]>({ reducer: (prev, next) => [...prev, ...next], default: () => [] }),
});
const sendAll = new StateGraph(State)
  .addNode("tutor", async (s) => {
    const { reply, tokens } = await chat(s.messages);
    console.log(`[LLM]  보낸 messages ${s.messages.length}개 · 입력 토큰 ${tokens}`);
    return { messages: [reply] };
  })
  .addEdge(START, "tutor").addEdge("tutor", END)
  .compile();
await showGraph(sendAll);
const a = await sendAll.invoke({ messages: history });
console.log(`[STATE] messages ${a.messages.length}개\n[답]  `, a.messages.at(-1).content);

**돌려 보면** messages 8개를 보냈고, 입력 토큰이 200개쯤입니다.

이번에는 노드 안에서 **골라서** 보냅니다. system 메시지와 가장 최근 메시지 하나만 보냅니다. State 는 고치지 않습니다. 모델에게 보낼 목록만 노드가 따로 만듭니다.

In [ ]:
// ② 골라서 보내기: system + 최근 n 개. State 는 그대로 전부 남는다
function recent(messages: any[], n: number) {
  return [messages[0], ...messages.slice(1).slice(-n)];
}
const sendRecent = new StateGraph(State)
  .addNode("tutor", async (s) => {
    const picked = recent(s.messages, 1);
    const { reply, tokens } = await chat(picked);
    console.log(`[LLM]  보낸 messages ${picked.length}개 · 입력 토큰 ${tokens}  (State 에는 ${s.messages.length}개)`);
    return { messages: [reply] };
  })
  .addEdge(START, "tutor").addEdge("tutor", END)
  .compile();
await showGraph(sendRecent);
const b = await sendRecent.invoke({ messages: history });
console.log(`[STATE] messages ${b.messages.length}개\n[답]  `, b.messages.at(-1).content);

**돌려 보면** 보낸 messages 는 2개, 입력 토큰은 50개쯤으로 줄었습니다. 결론을 묻는 질문에는 여전히 잘 답합니다. State 에는 대화가 그대로 있습니다.

그런데 보내지 않은 대화는 모델이 **모릅니다.** 맨 처음에 무엇을 물었는지 물어보겠습니다.

In [ ]:
// 대신, 보내지 않은 앞 대화는 모델이 모른다
const followUp = [...b.messages, { role: "user", content: "내가 맨 처음에 뭘 물어봤지?" }];
const c = await sendRecent.invoke({ messages: followUp });
console.log("[답]  ", c.messages.at(-1).content);
const d = await sendAll.invoke({ messages: followUp });
console.log("[답]  ", d.messages.at(-1).content);

**돌려 보면** 골라서 보낸 쪽은 처음 질문을 모릅니다. 전부 보낸 쪽은 "IELTS 쓰기에서 서론은 어떻게 써?" 라고 바로 답합니다.

그래서 무엇을 남기고 뺄지는 우리가 정하는 설계입니다. 개수 대신 토큰 수에 맞춰 자르는 방법도 있습니다(`@langchain/core` 의 `trimMessages`). 그리고 하나 더, 오래된 대화를 **요약해서** 남기는 방법이 있습니다.

## ② 요약하는 튜터

이번에는 State 에 `summary` 필드를 하나 더 둡니다. messages 는 앞 노트북에서 본 `MessagesAnnotation` 을 그대로 씁니다. 오래된 메시지를 `RemoveMessage` 로 지우려고요.

`MessagesAnnotation` 의 messages 에는 LangChain 메시지 객체가 들어 있습니다 (`state-reducers` 노트북 ③ 에서 본 HumanMessage · AIMessage). 그런데 우리 `chat()` 함수는 오전부터 써 온 `{ role, content }` 모양을 받습니다. 그래서 모델에게 보낼 때 `toApi` 로 모양을 바꿔 보냅니다.

In [ ]:
// chatText(): 앞 노트북과 같다. 다만 State 의 messages 가 LangChain 메시지 객체라서, 보낼 때 { role, content } 로 바꾼다
const ROLE: Record<string, string> = { human: "user", ai: "assistant", system: "system" };
const toApi = (m: any) => ({ role: ROLE[m.getType()], content: m.content });
async function callModelText(messages: any[]) {
  const res = await fetch(`${BASE_URL}/chat/completions`, {
    method: "POST",
    headers: { "Content-Type": "application/json", Authorization: `Bearer ${API_KEY}` },
    body: JSON.stringify({ model: MODEL, messages, ...OPTIONS }),
  });
  return ((await res.json()).choices[0].message.content ?? "").trim();
}
const chatText = asLLM(callModelText, "chatText");   // LangSmith 에 모델 호출로 남긴다

In [ ]:
// State: messages (MessagesAnnotation 그대로 — 붙이기 · 같은 id 덮어쓰기 · RemoveMessage 지우기) + summary 필드 하나
const ChatState = Annotation.Root({
  ...MessagesAnnotation.spec,
  summary: Annotation<string>(),
});
const TUTOR = "너는 IELTS 쓰기를 돕는 영어 튜터다. 한국어로 두세 문장으로 짧게 답한다.";

노드는 둘입니다.

- **tutor**: 요약이 있으면 system 메시지에 붙여서, 남아 있는 messages 와 함께 모델에게 보냅니다
- **summarize**: 지금까지의 대화로 요약을 씁니다. 요약이 이미 있으면 새 내용을 더해 고쳐 씁니다. 그리고 최근 2개만 남기고 나머지 메시지는 `RemoveMessage` 로 지웁니다

갈림길은 하나입니다. messages 가 6개를 넘으면 summarize 로, 아니면 끝.

In [ ]:
// 노드 ① tutor: 요약이 있으면 system 에 붙여서, 남아 있는 messages 와 함께 보낸다
async function tutor(state: typeof ChatState.State) {
  const system = state.summary ? `${TUTOR}

지금까지 대화 요약: ${state.summary}` : TUTOR;
  const reply = await chatText([{ role: "system", content: system }, ...state.messages.map(toApi)]);
  console.log(`[LLM]  보낸 messages ${state.messages.length + 1}개${state.summary ? " (요약 포함)" : ""}`);
  return { messages: [{ role: "assistant", content: reply }] };
}

// 노드 ② summarize: 요약을 쓰고(있으면 늘리고), 최근 2개만 남기고 나머지는 지운다
async function summarize(state: typeof ChatState.State) {
  const ask = state.summary
    ? `지금까지의 요약: ${state.summary}

위 대화의 새 내용을 더해서 요약을 고쳐 써라.`
    : "위 대화를 요약하라.";
  const summary = await chatText([...state.messages.map(toApi), { role: "user", content: `${ask} 학생 이름, 시험, 목표, 물어본 것을 빠뜨리지 말고 한국어 세 문장 이내로.` }]);
  const remove = state.messages.slice(0, -2).map((m) => new RemoveMessage({ id: m.id! }));
  console.log(`▶ summarize  요약을 쓰고, messages ${remove.length}개를 지운다`);
  return { summary, messages: remove };
}

// 갈림길: messages 가 6개를 넘으면 요약하러, 아니면 끝
function shouldSummarize(state: typeof ChatState.State) {
  return state.messages.length > 6 ? "summarize" : END;
}

그래프로 잇고, checkpointer 를 달아 compile 합니다. 여러 번 나눠 묻는 대화라서 thread 가 있어야 앞 대화가 이어집니다.

In [ ]:
const builder = new StateGraph(ChatState)
  .addNode("tutor", tutor)
  .addNode("summarize", summarize)
  .addEdge(START, "tutor")
  .addConditionalEdges("tutor", shouldSummarize, ["summarize", END])
  .addEdge("summarize", END);
const bot = builder.compile({ checkpointer: new MemorySaver() });
await showGraph(bot);
const thread = { configurable: { thread_id: "jimin" } };

async function say(text: string) {
  const out = await bot.invoke({ messages: [{ role: "user", content: text }] }, thread);
  console.log(`[학생] ${text}
[튜터] ${out.messages.at(-1)?.content}`);
  console.log(`[STATE] messages ${out.messages.length}개 · summary ${out.summary ? "있음" : "없음"}
`);
}

지민이라는 학생이 네 번 묻습니다. 매번 State 에 messages 가 몇 개 남았는지, 요약이 있는지 찍어 보겠습니다.

In [ ]:
await say("안녕, 나는 지민이야. 다음 달에 IELTS 쓰기 시험을 봐.");
await say("목표 점수는 7.0 이야.");
await say("Task 2 서론은 어떻게 써?");
await say("본론은 몇 문단이 좋아?");

**돌려 보면** messages 가 2, 4, 6 개로 늘다가, 네 번째에서 8개가 되면서 summarize 가 돕니다. 6개를 지우고 최근 2개만 남았고, 이제 summary 가 있습니다.

요약을 보겠습니다.

In [ ]:
// 요약이 ChatState 에 남았다
console.log((await bot.getState(thread)).values.summary);

**돌려 보면** 지민이라는 이름, IELTS 쓰기 시험, 목표 7.0, 물어본 것들이 몇 문장으로 남아 있습니다.

메시지는 지워졌는데, 이름과 목표 점수를 기억할까요?

In [ ]:
// 오래된 메시지는 지워졌지만, 요약 덕분에 기억한다
await say("내 이름이랑 목표 점수 기억해?");

**돌려 보면** "지민" 과 "7.0" 을 기억합니다. 모델에게 간 messages 는 몇 개 안 되지만 요약이 system 에 같이 갔기 때문입니다. 대화가 계속 길어져도 State 는 일정한 크기로 유지됩니다.

## ③ 다시 켜면

마지막으로 하나 더. `MemorySaver` 는 이름처럼 **메모리에만** 저장합니다. 프로그램을 끄면 같이 사라집니다. 새 MemorySaver 로 compile 해서, 프로그램을 다시 켠 상황을 흉내 내 보겠습니다. thread_id 는 그대로 "jimin" 입니다.

In [ ]:
// MemorySaver 는 메모리에만 저장한다. 프로그램을 다시 켜면(= 새 MemorySaver) 같은 thread_id 라도 비어 있다
const restarted = builder.compile({ checkpointer: new MemorySaver() });
const again = await restarted.invoke({ messages: [{ role: "user", content: "내 이름이랑 목표 점수 기억해?" }] }, thread);
console.log("[튜터]", again.messages.at(-1)?.content);

**돌려 보면** 앞 대화가 없습니다. 모델이 "기억한다" 고 답하면서 다른 점수를 지어내기도 합니다. State 에는 아무것도 없는데도요.

오래 남겨야 하는 대화라면 파일이나 데이터베이스에 저장하는 checkpointer 를 씁니다. LangGraph 에는 SQLite, Postgres 용 checkpointer 가 있고, `compile({ checkpointer })` 에 넘기는 것만 바뀝니다. LangGraph Studio 도 자체 저장소에 thread 를 저장해서, 창을 닫았다 열어도 대화가 남아 있습니다.